# Test of S1-ARD processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-766

In [ ]:
# For testing, don't commit
import sys
sys.path.insert(0, "/home/jgaucher/projects/rspy/github/rs-demo/notebooks")
import resources.test_localhost

In [ ]:
# Experimental configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
experimental_config = {
    "local_cluster": {
        "enabled": True, # Use False to disable
        "n_workers": 4,
        "memory_limit": "10GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_s1ard(
    # image="ghcr.io/rs-python/rs-infra-core-dask-s1ard:my-custom-tag", # default tag is :latest
    scale=1,              # number of workers
    big_resources=False,  # provide more ram and cpu
    worker_cores=4,       # number of CPU per worker 
    worker_memory=12,     # memory per worker in GB
)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcess

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-danger">

Note: not implemented for now for S1-ARD.
</div>

In [ ]:
# NOTE: not implemented for now in S1-ARD
# for process in [DprProcess.S1ARD]:
#     tasktable: dict = dpr_client.get_process(process.value)
#     print(f"Tasktable for {process.value!r}:")
#     display(JSON(tasktable))
#     # print(json.dumps(tasktable, indent=2))

## Init environment for the processors

In [ ]:
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir="./config"
)
await dpr.init(local_secrets_file="./config/secrets.json")
    
# Same arguments for all tests
dpr_args = {
    "process": DprProcess.S1ARD,
    "payload_subpath": "s1-ard/demo_joborder.yaml",
    "experimental_config": experimental_config,
    # Payload env vars
    "N_WORKERS": len(dask_client_eopf.scheduler_info()["workers"]), # Number of dask workers
}

## Run processor

In [ ]:
# Clean the working dir in the s3 bucket ?
clean_working_dir = True

In [ ]:
# Run S1-ARD
if os.getenv("RSPY_FROM_CICD") != "1":
    s3_output_dir = osp.join(dpr.s3_output_dir, "s1ard")
    s3_working_dir = osp.join(dpr.s3_working_dir, "s1ard")
    await dpr.run(
        **dpr_args,
        s3_output_dir = s3_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "s1ard"),
        del_s3_working_dir = s3_working_dir if clean_working_dir else "",
        # Payload env vars
        OUTPUT_DIR = s3_output_dir,
        WORKING_DIR = s3_working_dir,
    )

## Shutdown cluster

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.